# A* Search Algorithm: 8-Puzzle Solver
## SCIS 432: Artificial Intelligence

In this notebook, you'll implement the A* search algorithm to solve the classic 8-puzzle problem, and then apply it to unscramble images!

**Learning Objectives:**
1. Understand A* as an informed search algorithm (best-first search with heuristics)
2. Implement and compare different heuristic functions
3. See how heuristics affect search efficiency
4. Apply A* to solve visual puzzles (scrambled images)

**Structure:**
- **Part 0:** Setup and puzzle representation
- **Part 1:** Basic A* with simple heuristic (misplaced tiles)
- **Part 2:** Better heuristic (Manhattan distance)
- **Part 3:** Image puzzle variant (unscramble your selfie!)
- **Part 4:** Performance comparison

---

## Part 0: Puzzle Representation

The 8-puzzle is a 3x3 grid with tiles numbered 1-8 and one empty space (represented as 0).

**Goal:** Arrange tiles from scrambled state to goal state.
```
Initial State:        Goal State:
1 2 3                 1 2 3
4 0 5        →        4 5 6
7 8 6                 7 8 0
```

We'll represent the puzzle as a flat list of 9 elements:
- `[1, 2, 3, 4, 0, 5, 7, 8, 6]` for the initial state above
- `[1, 2, 3, 4, 5, 6, 7, 8, 0]` for the goal state

**Why A* over BFS/DFS?**
- BFS explores ~180,000 states for hard puzzles
- A* with good heuristic: ~1,000 states
- **~180x faster!**

In [1]:
import heapq
from collections import deque
import time

# Goal state for the 8-puzzle
GOAL_STATE = [1, 2, 3, 4, 5, 6, 7, 8, 0]

def print_puzzle(state):
    """Display puzzle in 3x3 grid format."""
    for i in range(0, 9, 3):
        row = state[i:i+3]
        print(" ".join(str(x) if x != 0 else " " for x in row))
    print()

def get_blank_position(state):
    """Return the index of the blank tile (0)."""
    return state.index(0)

def get_neighbors(state):
    """
    Get all valid neighbor states from current state.

    Returns:
        List of (new_state, move_description) tuples
    """
    neighbors = []
    blank_pos = get_blank_position(state)
    row, col = blank_pos // 3, blank_pos % 3

    # Possible moves: up, down, left, right
    moves = [
        (-1, 0, "UP"),    # Move blank up (tile moves down)
        (1, 0, "DOWN"),   # Move blank down (tile moves up)
        (0, -1, "LEFT"),  # Move blank left (tile moves right)
        (0, 1, "RIGHT")   # Move blank right (tile moves left)
    ]

    for dr, dc, move_name in moves:
        new_row, new_col = row + dr, col + dc

        # Check if move is valid
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_pos = new_row * 3 + new_col
            # Create new state by swapping blank with adjacent tile
            new_state = state.copy()
            new_state[blank_pos], new_state[new_pos] = new_state[new_pos], new_state[blank_pos]
            neighbors.append((new_state, move_name))

    return neighbors

# Test the functions
test_state = [1, 2, 3, 4, 0, 5, 7, 8, 6]
print("Test state:")
print_puzzle(test_state)
print("Blank position:", get_blank_position(test_state))
print("\nPossible moves:")
for neighbor, move in get_neighbors(test_state):
    print(f"{move}:")
    print_puzzle(neighbor)

Test state:
1 2 3
4   5
7 8 6

Blank position: 4

Possible moves:
UP:
1   3
4 2 5
7 8 6

DOWN:
1 2 3
4 8 5
7   6

LEFT:
1 2 3
  4 5
7 8 6

RIGHT:
1 2 3
4 5  
7 8 6



---
## Part 1: A* Search with Simple Heuristic

### A* Algorithm Overview

**A* = Best-First Search + Cost Function**

**Cost Function:** `f(n) = g(n) + h(n)`
- `g(n)` = cost from start to current node (number of moves so far)
- `h(n)` = **heuristic** estimate of cost from current node to goal
- `f(n)` = estimated total cost

**Key Insight:** Always explore the node with lowest `f(n)` first!

### Heuristic #1: Misplaced Tiles

Count how many tiles are not in their goal position.

Example:
```
Current: [1, 2, 3, 4, 0, 5, 7, 8, 6]
Goal:    [1, 2, 3, 4, 5, 6, 7, 8, 0]
Misplaced: 0, 5, 6 → h(n) = 3
```

**TODO: Implement the heuristic and A* search**

## Fill-in Below

In [ ]:
def heuristic_misplaced(state, goal=GOAL_STATE):
    """
    Heuristic function: count misplaced tiles.

    Args:
        state: Current puzzle state
        goal: Goal puzzle state

    Returns:
        Number of tiles not in their goal position (excluding blank)

    Hint: Compare each position; don't count the blank tile (0)
    """
    # TODO: Count how many tiles are in wrong positions
    # Remember to skip the blank tile (0)

    pass  # Replace with your code


# Test your heuristic
test_state = [1, 2, 3, 4, 0, 5, 7, 8, 6]
print(f"Test state:")
print_puzzle(test_state)
print(f"Misplaced tiles heuristic: {heuristic_misplaced(test_state)}")
print(f"Expected: 3 (tiles 5, 6, and 0 are misplaced)")

### PuzzleNode Class (Use This!)

In [ ]:
class PuzzleNode:
    """
    Node in the search tree.

    Attributes:
        state: Current puzzle configuration
        parent: Parent node (for reconstructing path)
        move: Move that led to this state
        g: Cost from start (number of moves)
        h: Heuristic value
        f: Total cost (g + h)
    """
    def __init__(self, state, parent=None, move=None, g=0, h=0):
        self.state = state
        self.parent = parent
        self.move = move
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, other):
        """For priority queue comparison."""
        return self.f < other.f

    def __eq__(self, other):
        """For checking if states are equal."""
        return self.state == other.state

    def __hash__(self):
        """For using in sets."""
        return hash(tuple(self.state))


def reconstruct_path(node):
    """
    Reconstruct path from start to goal.

    Args:
        node: Goal node

    Returns:
        List of (state, move) tuples from start to goal
    """
    path = []
    while node.parent is not None:
        path.append((node.state, node.move))
        node = node.parent
    path.append((node.state, None))  # Start state
    return list(reversed(path))

## Fill-in Below!

In [ ]:
def a_star_search(start_state, goal_state, heuristic_func):
    """
    A* search algorithm.

    Args:
        start_state: Initial puzzle state
        goal_state: Goal puzzle state
        heuristic_func: Heuristic function to use

    Returns:
        Tuple of (path, nodes_explored) where path is list of (state, move) tuples
        Returns (None, nodes_explored) if no solution found

    Algorithm:
        1. Initialize priority queue with start node
        2. Keep track of explored states (to avoid revisiting)
        3. While queue not empty:
           a. Pop node with lowest f value
           b. If goal reached, return path
           c. Mark as explored
           d. For each neighbor:
              - If not explored, compute f = g + h and add to queue
        4. Return None if queue empty (no solution)

    Hint: Use heapq for priority queue:
        - heapq.heappush(queue, node)
        - node = heapq.heappop(queue)
    """

    # TODO: Initialize
    # - Create start node with g=0, h=heuristic_func(start_state)
    # - Create priority queue (list) and push start node
    # - Create explored set to track visited states
    # - Create nodes_explored counter


    # TODO: Main A* loop
    # While queue is not empty:
    #   - Pop node with lowest f value
    #   - Increment nodes_explored
    #   - If goal state reached, return (reconstruct_path(node), nodes_explored)
    #   - Add state to explored set
    #   - For each neighbor state:
    #       - If not in explored:
    #           - Create child node with:
    #               g = current.g + 1
    #               h = heuristic_func(neighbor_state)
    #           - Push child node to queue

    pass  # Replace with your code

## Test Code

In [ ]:
# Test A* on a simple puzzle
easy_puzzle = [1, 2, 3, 4, 5, 6, 0, 7, 8]  # Only 2 moves from goal

print("Solving easy puzzle:")
print_puzzle(easy_puzzle)

solution, nodes = a_star_search(easy_puzzle, GOAL_STATE, heuristic_misplaced)

if solution:
    print(f"Solution found in {len(solution) - 1} moves!")
    print(f"Nodes explored: {nodes}\n")

    print("Solution path:")
    for i, (state, move) in enumerate(solution):
        if move:
            print(f"Move {i}: {move}")
        else:
            print(f"Start state:")
        print_puzzle(state)
else:
    print("No solution found!")

---
## Part 2: Better Heuristic - Manhattan Distance

**Problem with misplaced tiles:** It underestimates the true cost.

Example: Tile 8 is one position off → counts as 1, but might need multiple moves to reach goal.

**Manhattan Distance:** Sum of horizontal + vertical distances each tile must move.
```
Current:     Goal:
1 2 3        1 2 3
4 0 5        4 5 6
7 8 6        7 8 0

Tile 5: needs to move right 1 → distance = 1
Tile 6: needs to move right 1, down 1 → distance = 2
Tile 8: needs to move right 1, down 1 → distance = 2
Manhattan distance = 1 + 2 + 2 = 5
```

**Why it's better:** More accurate estimate → explores fewer wrong paths!

**TODO: Implement Manhattan distance heuristic**

## Fill-in Heuristic function

In [ ]:
def heuristic_manhattan(state, goal=GOAL_STATE):
    """
    Heuristic function: Manhattan distance.

    Sum of distances each tile must move (horizontally + vertically).

    Args:
        state: Current puzzle state
        goal: Goal puzzle state

    Returns:
        Sum of Manhattan distances for all tiles (excluding blank)

    Hint:
        - For each tile (except 0), find its current position and goal position
        - Convert index to (row, col): row = index // 3, col = index % 3
        - Distance = |current_row - goal_row| + |current_col - goal_col|
    """
    # TODO: Implement Manhattan distance
    # For each tile value (1-8):
    #   - Find its current position (row, col)
    #   - Find its goal position (row, col)
    #   - Add |current_row - goal_row| + |current_col - goal_col| to total

    pass  # Replace with your code


# Test your heuristic
test_state = [1, 2, 3, 4, 0, 5, 7, 8, 6]
print(f"Test state:")
print_puzzle(test_state)
print(f"Manhattan distance: {heuristic_manhattan(test_state)}")
print(f"Misplaced tiles: {heuristic_misplaced(test_state)}")
print(f"\nManhattan should be ≥ misplaced tiles (more accurate estimate)")

## Compare Heuristics

In [ ]:
# Compare both heuristics on a harder puzzle
medium_puzzle = [1, 2, 3, 4, 5, 6, 7, 0, 8]  # 1 move from goal, but let's try harder one
hard_puzzle = [7, 2, 4, 5, 0, 6, 8, 3, 1]    # ~20 moves from goal

print("Comparing heuristics on hard puzzle:")
print_puzzle(hard_puzzle)

# A* with misplaced tiles
start_time = time.time()
solution1, nodes1 = a_star_search(hard_puzzle, GOAL_STATE, heuristic_misplaced)
time1 = time.time() - start_time

# A* with Manhattan distance
start_time = time.time()
solution2, nodes2 = a_star_search(hard_puzzle, GOAL_STATE, heuristic_manhattan)
time2 = time.time() - start_time

print("\n" + "="*60)
print("RESULTS:")
print("="*60)
print(f"Misplaced Tiles Heuristic:")
print(f"  Solution length: {len(solution1) - 1 if solution1 else 'N/A'} moves")
print(f"  Nodes explored: {nodes1}")
print(f"  Time: {time1:.4f} seconds")
print()
print(f"Manhattan Distance Heuristic:")
print(f"  Solution length: {len(solution2) - 1 if solution2 else 'N/A'} moves")
print(f"  Nodes explored: {nodes2}")
print(f"  Time: {time2:.4f} seconds")
print()
print(f"Manhattan explored {((nodes1 - nodes2) / nodes1 * 100):.1f}% fewer nodes!")

---
## Part 3: Image Puzzle - Unscramble Your Selfie!

Now for the fun part! We'll apply the same A* algorithm to solve scrambled images.

**How it works:**
1. Upload an image (your selfie, pet, meme, etc.)
2. Split it into a 3x3 grid (9 tiles)
3. Scramble the tiles (like the 8-puzzle)
4. Use A* to solve it
5. Animate the solution!

**The cool part:** The algorithm is the same, we just change the visualization!

## Image Upload Helper

In [ ]:
# Install required packages (only needed in Colab)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

def upload_image():
    """
    Helper function to upload an image in Google Colab.

    Returns:
        PIL Image object

    How to use:
        1. Run this cell
        2. Click "Choose Files" button
        3. Select an image from your computer (jpg, png, etc.)
        4. Wait for upload to complete
    """
    if IN_COLAB:
        print("Click 'Choose Files' to upload your image!")
        print("Tip: Square images work best (we'll crop to square)")
        print()
        uploaded = files.upload()

        if uploaded:
            # Get the first uploaded file
            filename = list(uploaded.keys())[0]
            img = Image.open(filename)
            print(f"Uploaded: {filename}")
            print(f"   Size: {img.size}")
            return img
        else:
            print("No file uploaded")
            return None
    else:
        # If not in Colab, use a sample image or file dialog
        print("Not in Colab. Please provide image path:")
        path = input("Image path: ")
        return Image.open(path)

# Run this to upload your image!
# Uncomment the line below when ready:
# my_image = upload_image()

## Provided Image Puzzle Functions

In [ ]:
def prepare_puzzle_image(img, tile_size=150):
    """
    Prepare image for puzzle by making it square and resizing.

    Args:
        img: PIL Image
        tile_size: Size of each tile in pixels

    Returns:
        Prepared PIL Image (square, 3*tile_size x 3*tile_size)
    """
    # Convert to RGB if needed
    img = img.convert('RGB')

    # Crop to square (center crop)
    width, height = img.size
    min_dim = min(width, height)
    left = (width - min_dim) // 2
    top = (height - min_dim) // 2
    img = img.crop((left, top, left + min_dim, top + min_dim))

    # Resize to tile_size * 3
    img = img.resize((tile_size * 3, tile_size * 3), Image.Resampling.LANCZOS)

    return img


def split_image_into_tiles(img):
    """
    Split image into 9 tiles (3x3 grid).

    Args:
        img: PIL Image (must be square)

    Returns:
        List of 9 PIL Image tiles
    """
    width, height = img.size
    tile_width = width // 3
    tile_height = height // 3

    tiles = []
    for row in range(3):
        for col in range(3):
            left = col * tile_width
            top = row * tile_height
            right = left + tile_width
            bottom = top + tile_height

            tile = img.crop((left, top, right, bottom))
            tiles.append(tile)

    return tiles


def display_puzzle_state(tiles, state, title="Puzzle State"):
    """
    Display puzzle state as image.

    Args:
        tiles: List of 9 PIL Image tiles
        state: Puzzle state (list of 9 numbers)
        title: Title for the display
    """
    # Create blank canvas
    tile_size = tiles[0].size[0]
    canvas = Image.new('RGB', (tile_size * 3, tile_size * 3), 'black')

    # Place tiles according to state
    for i, tile_num in enumerate(state):
        if tile_num != 0:  # Skip blank tile
            row = i // 3
            col = i % 3
            # tile_num is 1-indexed, tiles list is 0-indexed
            canvas.paste(tiles[tile_num - 1], (col * tile_size, row * tile_size))

    # Display
    plt.figure(figsize=(6, 6))
    plt.imshow(canvas)
    plt.title(title, fontsize=16)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def create_scrambled_puzzle(tiles, difficulty='medium'):
    """
    Create a solvable scrambled puzzle.

    Args:
        tiles: List of tiles
        difficulty: 'easy' (5-10 moves), 'medium' (15-20 moves), 'hard' (25-30 moves)

    Returns:
        Scrambled state (guaranteed solvable)
    """
    import random

    # Start from goal state
    state = GOAL_STATE.copy()

    # Number of random moves
    num_moves = {'easy': 7, 'medium': 18, 'hard': 27}[difficulty]

    # Make random moves
    for _ in range(num_moves):
        neighbors = get_neighbors(state)
        state, _ = random.choice(neighbors)

    return state

## Create and Solve Image Puzzle

In [ ]:
# Example workflow (uncomment when you have an image)
"""
# 1. Upload your image (run Cell 13 first)
my_image = upload_image()

# 2. Prepare image
prepared_img = prepare_puzzle_image(my_image, tile_size=150)
tiles = split_image_into_tiles(prepared_img)

# 3. Show original (solved) puzzle
print("Original image (solved puzzle):")
display_puzzle_state(tiles, GOAL_STATE, "Goal State - Solved!")

# 4. Create scrambled puzzle
scrambled_state = create_scrambled_puzzle(tiles, difficulty='medium')
print("\nScrambled puzzle:")
display_puzzle_state(tiles, scrambled_state, "Scrambled - Can you solve it?")

# 5. Solve with A*
print("\nSolving with A* (Manhattan distance)...")
solution, nodes = a_star_search(scrambled_state, GOAL_STATE, heuristic_manhattan)

if solution:
    print(f"Solution found!")
    print(f"   Moves: {len(solution) - 1}")
    print(f"   Nodes explored: {nodes}")

    # 6. Show solution steps (optional - shows every state)
    show_all = input("\nShow all solution steps? (y/n): ").lower() == 'y'

    if show_all:
        for i, (state, move) in enumerate(solution):
            title = f"Step {i}/{len(solution)-1}"
            if move:
                title += f" - Move: {move}"
            else:
                title += " - Start"
            display_puzzle_state(tiles, state, title)
else:
    print("No solution found!")
"""

print("Uncomment and run the code above after uploading your image!")

## Animated Solution

In [ ]:
def animate_solution(tiles, solution, delay=0.5):
    """
    Animate the solution with smooth transitions.

    Args:
        tiles: List of image tiles
        solution: Solution path from A*
        delay: Seconds between frames
    """
    from IPython.display import clear_output
    import time as t

    print(f"Animating solution ({len(solution)-1} moves)...\n")

    for i, (state, move) in enumerate(solution):
        clear_output(wait=True)

        title = f"Step {i}/{len(solution)-1}"
        if move:
            title += f" - Move: {move}"
        else:
            title = "Start State"

        display_puzzle_state(tiles, state, title)

        if i < len(solution) - 1:  # Don't wait after last frame
            t.sleep(delay)

    print("\nPuzzle solved!")

# Example usage (uncomment after solving a puzzle):
"""
animate_solution(tiles, solution, delay=0.3)
"""

---
## Reflection Questions

1. **What makes a good heuristic for A*?**
   - Hint: Think about admissibility (never overestimate) and consistency

2. **Why does Manhattan distance explore fewer nodes than misplaced tiles?**

3. **Is the solution from A* always optimal (shortest path)?** Why or why not?

4. **What's the time complexity of A*?** How does it compare to BFS?

5. **How would you modify this for a 15-puzzle (4x4 grid)?**
   - Would the same heuristics work?
   - Would you need to change the code structure?

6. **What other problems could you solve with A*?**
   - Pathfinding in video games?
   - Route planning in GPS?
   - Robotics?

---

## Extensions (Optional Challenges)

Try these to deepen your understanding:

1. **Better visualization:** Add arrows showing which tile moved
2. **Pattern database heuristic:** Pre-compute exact costs for subsets of tiles
3. **Iterative Deepening A* (IDA*):** Memory-efficient variant
4. **15-puzzle:** Extend to 4x4 grid (much harder!)
5. **Custom heuristic:** Combine Manhattan + Linear Conflict
6. **Solvability checker:** Detect unsolvable states before searching
7. **Interactive game:** Let user scramble, then solve with A*
8. **Different goal states:** Arrange tiles in different patterns

---

## Solutions Checklist

Before checking solutions, verify your implementation:

**Heuristic Functions:**
- [ ] Misplaced tiles returns correct count (test cases provided)
- [ ] Manhattan distance ≥ misplaced tiles (always true)
- [ ] Both heuristics return 0 for goal state
- [ ] Neither heuristic overestimates (admissible)

**A* Search:**
- [ ] Always finds optimal solution (shortest path)
- [ ] Manhattan explores significantly fewer nodes than misplaced tiles
- [ ] A* explores WAY fewer nodes than BFS (often 100x fewer)
- [ ] Solution path is valid (each step is a legal move)
- [ ] Works correctly on goal state (returns immediately)

**Image Puzzle:**
- [ ] Image loads and displays correctly
- [ ] Scrambled puzzle is solvable
- [ ] Solution successfully unscrambles the image
- [ ] Animation shows smooth progression

---

## Tips for Success

**Debugging A*:**
- Print f, g, h values to verify calculations
- Check if explored set is working (shouldn't revisit states)
- Verify priority queue is using f values correctly

**Heuristic Design:**
- Test on known puzzles with known solution lengths
- Heuristic should never be negative
- h(goal) should always be 0

**Image Puzzles:**
- Use square images for best results
- Start with easy scrambles to verify correctness
- Medium difficulty is good for demonstrations

Happy coding!